# Notebook 02 — RLM Experiments

This notebook is organised as a presentation-friendly walkthrough of Recursive
Language Models (RLMs), starting with a simple code-execution example and then
moving into recursive decomposition.

| Section | Concept demonstrated |
|---|---|
| Intro-A | **Letter counting via Python** — let the agent write code instead of guessing |
| Intro-B | **From tool use to recursion** — why code execution motivates RLMs |
| 2-A | **Needle-in-a-Haystack** — find a hidden value in a long text |
| 2-B | **Hierarchical summarisation** — `llm_call` for leaf summaries, `rlm_call` for recursive decomposition |
| 2-C | **Max depth & base case** — observe the fallback to plain LLM |
| 2-D | **Prompt tracing** — inspect the exact messages sent to the LLM server |
| 2-E | **Saving metadata** — persist call-tree and prompt-trace payloads |

## Two sub-call tools

The REPL now exposes two tools that mirror the paper's `llm_query` / `rlm_query` distinction:

| Tool | Behaviour | Use when |
|---|---|---|
| `llm_call(sub_task, context_slice)` | Direct, non-recursive LLM call | Chunk is small enough to answer in one shot |
| `rlm_call(sub_task, context_slice)` | Recursive sub-agent with its own REPL | Sub-task may need further decomposition |


## Intro-A: Why start with letter counting?

A good opening example is a task that humans solve reliably, but language models
often solve inconsistently when they rely on next-token prediction alone.

Use this framing in the session:

1. Ask the audience to count a specific letter in a word such as `strawberry`.
2. Point out that a plain LLM may guess or pattern-match.
3. Then show that once the model is allowed to write Python, the task becomes deterministic.

That is the bridge to RLMs: instead of forcing the model to do everything in one
shot inside a token stream, we let it externalise reasoning into code and later
into recursive subcalls.

## Setup

In [ ]:
import os
import sys
import json
import random
import textwrap

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

LLAMA_BASE_URL = os.environ.get('LLAMA_BASE_URL', 'http://host-gateway:1337/v1')
LLAMA_MODEL    = os.environ.get('LLAMA_MODEL',    'local-model')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'not-needed')

from rlm_smolagent import RLMAgent

def make_agent(max_depth=2, max_steps=10, verbose=False, capture_prompt_traces=False):
    return RLMAgent(
        base_url=LLAMA_BASE_URL,
        model_name=LLAMA_MODEL,
        api_key=OPENAI_API_KEY,
        max_depth=max_depth,
        max_steps=max_steps,
        verbose=verbose,
        capture_prompt_traces=capture_prompt_traces,
    )

print('Setup complete.')

## Helper: pretty-print the call tree

In [ ]:
def print_tree(node: dict, indent: int = 0) -> None:
    """Recursively print a call-tree dictionary produced by RLMAgent."""
    prefix = '  ' * indent
    depth  = node.get('depth', '?')
    dur    = node.get('duration_s', '?')
    task   = node.get('task_preview', '')   # key is task_preview (not prompt_preview)
    resp   = node.get('response_preview', '')
    print(f"{prefix}[depth={depth}] ({dur}s)")
    print(f"{prefix}  task    : {task}")
    print(f"{prefix}  response: {resp}")
    for child in node.get('children', []):
        print_tree(child, indent + 1)


## Helper: inspect prompt traces

`capture_prompt_traces=True` records the exact message payloads sent to the
OpenAI-compatible server for each CodeAgent step and each plain fallback call.
These helpers make the captured data easier to inspect in the notebook.

In [ ]:
def count_llm_requests(node: dict) -> int:
    total = len(node.get('llm_requests', []))
    for child in node.get('children', []):
        total += count_llm_requests(child)
    return total

def print_prompt_trace(result) -> None:
    print(result.format_prompt_trace())

def print_request_summary(result) -> None:
    requests = result.iter_llm_requests()
    print(f'Total outbound LLM requests: {len(requests)}')
    for index, request in enumerate(requests, start=1):
        tools = ', '.join(request.get('tool_names', [])) or 'none'
        print(
            f"{index:02d}. depth={request.get('depth')} "
            f"phase={request.get('phase')} tools={tools}"
        )

---
## Intro-B: Opening demo — let the agent write Python for counting

In this demo, we intentionally use a task that many LLMs answer unreliably when
they guess from tokens alone: counting letters inside a word.

The goal is not recursion yet. The goal is to show a more general principle:
when the model can use a Python REPL, it can convert a brittle cognitive task
into an exact computation.

In [ ]:
count_word = 'strawberry'
target_letter = 'r'

count_agent = make_agent(
    max_depth=1,
    max_steps=8,
    verbose=False,
    capture_prompt_traces=True,
 )

# Short task — no separate context needed for a simple counting exercise.
count_result = count_agent.completion(
    task=(
        f"Count how many times the letter '{target_letter}' appears in the word"
        f" '{count_word}'.\n\n"
        "Important: do not answer from intuition.\n"
        "Use Python code in the REPL to compute the answer exactly, then return:\n"
        "- the final count\n"
        "- a one-sentence explanation of why using code is more reliable here\n"
        "\nKeep the response short."
    ),
)

print('=== Counting Demo Answer ===')
print(count_result.response)

print('\n=== Prompt Summary ===')
print_request_summary(count_result)

In [ ]:
print('\n=== Prompt Trace ===')
print_prompt_trace(count_result)

print('\n=== Raw First Request ===')
print(json.dumps(count_result.iter_llm_requests()[0], indent=2))

## Intro-C: From code execution to RLM

The counting demo shows the first step: the model can offload an exact subtask
to Python instead of approximating it in natural language.

RLMs extend this idea one level further:

1. The model can decide that a task is too large or too complex for one pass.
2. It has two sub-call tools available inside the REPL:
   - `llm_call(sub_task, context_slice)` — a direct, non-recursive LLM call (fast).
   - `rlm_call(sub_task, context_slice)` — a recursive sub-agent call (for complex subtasks).
3. Each `rlm_call` spawns a child agent with its own REPL that can decompose further.
4. The parent agent aggregates the child results into a final answer.

So the core idea is not only tool use. It is tool use plus **self-decomposition** —
and the model freely decides which tool to use and how to split the context.


---
## Experiment 2-A: Needle-in-a-Haystack

We hide a secret word inside a long passage of random text and ask the RLM to
find it.  Crucially, the haystack is passed as `context=` — it is stored as the
Python variable `rlm_context` in the REPL, **not embedded in the prompt string**.

The model should:
1. Use Python to split `rlm_context` into halves.
2. Call `rlm_call(sub_task, half)` for each half.
3. Report the result from the half that contained the needle.

> This is the canonical "Hello World" example from the RLM paper.

In [ ]:
random.seed(7)

WORDS = (
    "the quick brown fox jumps over the lazy dog "
    "lorem ipsum dolor sit amet consectetur adipiscing elit "
).split()

def make_haystack(n_words: int = 800, needle_word: str = "SECRET_42") -> str:
    words = [random.choice(WORDS) for _ in range(n_words)]
    insert_pos = random.randint(n_words // 4, 3 * n_words // 4)
    words.insert(insert_pos, needle_word)
    return ' '.join(words)

needle  = "SECRET_42"
haystack = make_haystack(n_words=800, needle_word=needle)

print(f'Haystack length : {len(haystack.split())} words')
print(f'Needle          : "{needle}"')

In [ ]:
agent = make_agent(max_depth=2, max_steps=12, verbose=True)

# The needle/haystack task:
#   task    = short description (no haystack content here)
#   context = the actual haystack text → available as `rlm_context` in the REPL
result = agent.completion(
    task=(
        f'The text in `rlm_context` contains the word "{needle}" exactly once.\n'
        'Find that word and return the 5 words that appear immediately before it.\n\n'
        'Strategy:\n'
        '  1. Use Python to split rlm_context into two halves.\n'
        '  2. Call rlm_call(sub_task, half) to search each half — '
             'pass the half as context_slice, NOT embedded in sub_task.\n'
        '  3. Return the context from whichever half found the needle.'
    ),
    context=haystack,   # stored as rlm_context in the REPL, not in the prompt
)

print('\n=== RLM Answer ===')
print(result.response)

In [ ]:
print('\n=== Call Tree ===')
print_tree(result.metadata['call_tree'])

---
## Experiment 2-B: Hierarchical Summarisation

We supply three short articles as `context=`.  The model receives `rlm_context`
as a Python variable containing the concatenated articles.  The task description
tells it to extract individual articles programmatically and delegate each to a
child call — the raw text is never repeated in the prompt string.

This experiment demonstrates the **two-tool distinction**:

- Use `llm_call()` for each per-article summary — the articles are small enough
  to answer in a single direct LLM call without spawning a full sub-agent.
- Use `rlm_call()` when a subtask itself needs recursive decomposition.

Expected flow:
1. `articles = rlm_context.split("===")` (or similar)
2. `llm_call("Summarise article A in one sentence", articles[0])` — direct call
3. Combine the three one-sentence summaries into a meta-summary.


In [ ]:
ARTICLES = [
    (
        "Article A: Climate Change",
        "Global temperatures have risen by approximately 1.1°C above pre-industrial levels. "
        "Scientists warn that without drastic emission cuts, warming could exceed 2°C by 2100, "
        "leading to more frequent extreme weather events, rising sea levels, and biodiversity loss. "
        "Renewable energy adoption and policy changes are cited as the most critical levers."
    ),
    (
        "Article B: Artificial Intelligence",
        "Large language models (LLMs) have transformed natural language processing. "
        "Models with hundreds of billions of parameters can now write code, compose essays, "
        "and solve mathematical problems. Concerns around hallucination, bias, and energy "
        "consumption remain active research areas. Recursive inference techniques are emerging "
        "as a way to extend LLM capabilities beyond their context window."
    ),
    (
        "Article C: Space Exploration",
        "NASA's Artemis programme aims to return humans to the Moon by the late 2020s. "
        "SpaceX's Starship, the largest rocket ever built, is central to lunar and eventual "
        "Mars missions. Private companies have begun to lower the cost of orbital access "
        "significantly, opening new possibilities for satellite deployment and space tourism."
    ),
]

articles_text = '\n\n'.join(f'=== {title} ===\n{body}' for title, body in ARTICLES)
print(articles_text)

In [ ]:
agent = make_agent(max_depth=2, max_steps=12, verbose=True)

# articles_text lives in `context`, not embedded in the prompt.
# Inside the REPL it is available as `rlm_context`.
result = agent.completion(
    task=(
        "`rlm_context` contains three articles separated by '===' markers.\n"
        "Step 1: Use Python to split rlm_context into individual articles.\n"
        "Step 2: Call llm_call() once per article (NOT rlm_call — these articles "
                "are small enough to summarise directly), passing the article text "
                "as context_slice, to get a one-sentence summary of each.\n"
        "Step 3: Combine the three summaries into a single 2-3 sentence "
                "meta-summary and return it."
    ),
    context=articles_text,  # stored as rlm_context in the REPL
)

print('\n=== Meta-Summary ===')
print(result.response)


In [ ]:
print('\n=== Call Tree ===')
print_tree(result.metadata['call_tree'])

---
## Experiment 2-C: Max Depth & Base Case

Set `max_depth=0` to force the agent to use the plain LLM fallback immediately.
This is useful for comparing the RLM answer vs. the naive single-shot answer.

In [ ]:
task = "Explain what a Recursive Language Model is in one paragraph."

# Plain single-shot (max_depth=0 → no recursion, falls back to plain LLM)
plain_agent = make_agent(max_depth=0, max_steps=5, verbose=False)
plain_result = plain_agent.completion(task=task)

# Full RLM (max_depth=2)
rlm_agent = make_agent(max_depth=2, max_steps=10, verbose=False)
rlm_result = rlm_agent.completion(task=task)

print('--- Plain (depth=0) ---')
print(plain_result.response)
print()
print('--- RLM (depth=2) ---')
print(rlm_result.response)

---
## Experiment 2-D: Prompt tracing and sub-agent instructions

This experiment turns on `capture_prompt_traces` so you can inspect the exact
messages sent to the LLM server at each recursive step.

What to look for:

1. The root agent prompt includes the system hint that teaches the model how to use
   both `llm_call()` (direct) and `rlm_call()` (recursive) tools.
2. `llm_call()` invocations produce `plain_completion` trace entries (no sub-REPL).
3. `rlm_call()` invocations produce `agent_step` entries at a deeper recursion level.
4. If a leaf call falls back to plain completion, that payload is captured too.


In [ ]:
trace_agent = make_agent(
    max_depth=2,
    max_steps=10,
    verbose=False,
    capture_prompt_traces=True,
 )

# No separate context needed — just a short task description.
# The two child calls demonstrate the difference between llm_call and rlm_call:
#   llm_call → direct, fast, produces a plain_completion trace entry
#   rlm_call → recursive, produces agent_step trace entries at depth+1
trace_result = trace_agent.completion(
    task=(
        "Explain how recursive summarisation works.\n\n"
        "Make exactly two sub-calls:\n"
        "  1. Use llm_call() to get a direct one-sentence definition of recursion.\n"
        "  2. Use rlm_call() to get a child agent's explanation of why aggregation is needed.\n"
        "Combine the two child answers into a short final explanation."
    ),
)

print('=== Final Answer ===')
print(trace_result.response)

print('\n=== Request Summary ===')
print_request_summary(trace_result)

print('\n=== Prompt Trace ===')
print_prompt_trace(trace_result)


In [ ]:
print('\n=== Call Tree ===')
print_tree(trace_result.metadata['call_tree'])

print('\n=== Raw Trace Payload (first request) ===')
first_request = trace_result.iter_llm_requests()[0]
print(json.dumps(first_request, indent=2))

print('\n=== Total Requests In Tree ===')
print(count_llm_requests(trace_result.metadata['call_tree']))

---
## Experiment 2-E: Saving and loading metadata

The `metadata` dict can be serialised to JSON for later analysis or
visualisation. When prompt tracing is enabled, the saved metadata also contains
the `llm_requests` payloads for each node in the recursive tree.

In [ ]:
import pathlib

log_dir = pathlib.Path('/workspace/logs')
log_dir.mkdir(parents=True, exist_ok=True)

log_file = log_dir / 'experiment_2d_prompt_trace.json'
log_file.write_text(json.dumps(trace_result.metadata, indent=2))

print(f'Metadata saved to {log_file}')

# Load and display
loaded = json.loads(log_file.read_text())
print_tree(loaded['call_tree'])
print()
print(f"Total saved LLM requests: {count_llm_requests(loaded['call_tree'])}")